# Phase 3 – Rhetorical Role Prediction (RR)
## Experiment 6: IA³ Multiplicative Scaling on Architecture 4 (ALiBi + SwiGLU + RMSNorm)

**Technique**: **IA³ (Infused Adapter by Inhibiting and Amplifying Inner Activations)**  
**Base Model**: Phase-1 Architecture 4 – `ALiBiSwiGLULM` (~105M params, Best Phase 1 Model PPL=23.66)  
**Decoder Checkpoint**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt`  
**Custom Tokenizer**: `/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json`  
**Dataset**: Hugging Face `Exploration-Lab/CS-781-Capstone` (Phase_3_RR)  

---

### Key Highlights
- **Ultra-Lightweight Parameter Adaptation**: Trains only three scaling vectors per layer ($l_k, l_v, l_{ff}$), taking under ~48K trainable parameters while retaining 100% of Arch 4's pretrained capacity.
- **Zip Archive Auto-Fix**: Eliminates `.format_version` loading errors.
- **Chunked Sentence Encoding**: Prevents CUDA OOM on long legal documents.
- **Document-Level CRF**: Models rhetorical role transition logic (e.g. `Fact` -> `Issue` -> `Argument`).

## 1. Setup & Environment

In [ ]:
import os, gc, json, math, random, time, zipfile, warnings
from collections import Counter, defaultdict
from dataclasses import dataclass
from typing import List, Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')

## 2. Paths & Hugging Face Dataset Download

In [ ]:
from huggingface_hub import login, hf_hub_download

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

if HF_TOKEN:
    login(token=HF_TOKEN)
    print("Hugging Face login successful.")

PHASE1_MODEL_PATH = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/decoder_lm.pt'
TOKENIZER_PATH    = '/kaggle/input/datasets/brindafadadu/cs781-phase1-arch4/custom_bpe.json'

REPO = "Exploration-Lab/CS-781-Capstone"
print("Downloading Phase 3 RR datasets from Hugging Face...")
TRAIN_CL = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_CL   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/CL_dev.json", repo_type="dataset", token=HF_TOKEN)
TRAIN_IT = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_train.json", repo_type="dataset", token=HF_TOKEN)
DEV_IT   = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/IT_dev.json", repo_type="dataset", token=HF_TOKEN)
LABEL_VOCAB_PATH = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Train/label_vocab.json", repo_type="dataset", token=HF_TOKEN)
TEST_CL  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/CL_test.json", repo_type="dataset", token=HF_TOKEN)
TEST_IT  = hf_hub_download(repo_id=REPO, filename="Phase_3_RR/Test/IT_test.json", repo_type="dataset", token=HF_TOKEN)

OUTPUT_DIR = '/kaggle/working/phase3_exp6_ia3_arch4'
os.makedirs(OUTPUT_DIR, exist_ok=True)

MAX_SEQ_LEN     = 256
CONTEXT_WINDOW  = 3
CHUNK_SIZE      = 16
LEARNING_RATE   = 4e-4
WEIGHT_DECAY    = 0.001
EPOCHS          = 12
GRAD_CLIP       = 1.0

print("IA3 paths & hyperparameters configured.")

## 3. Checkpoint Loader with Zip Archive Auto-Fix

In [ ]:
def load_checkpoint_safe(model_path):
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Checkpoint not found at: {model_path}")

    if os.path.isdir(model_path):
        files = os.listdir(model_path)
        pt_files = [f for f in files if f.endswith('.pt') or f.endswith('.pth')]
        if pt_files: model_path = os.path.join(model_path, pt_files[0])
        else: raise FileNotFoundError(f"No .pt file found in: {model_path}")

    try:
        return torch.load(model_path, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f"Direct load exception ({str(e)[:80]}). Running zip auto-fix...")

    extract_dir = "/kaggle/working/extracted_arch4_ckpt"
    fixed_pt    = "/kaggle/working/fixed_arch4_decoder_lm.pt"
    os.makedirs(extract_dir, exist_ok=True)

    with zipfile.ZipFile(model_path, 'r') as z: z.extractall(extract_dir)
    nested = [f for f in os.listdir(extract_dir) if f.endswith('.pt') and f != os.path.basename(model_path)]
    if nested: return torch.load(os.path.join(extract_dir, nested[0]), map_location='cpu', weights_only=False)

    with zipfile.ZipFile(fixed_pt, 'w', compression=zipfile.ZIP_DEFLATED) as z_out:
        for root, _, files in os.walk(extract_dir):
            for file in files:
                if file in [".format_version", os.path.basename(fixed_pt)]: continue
                full_path = os.path.join(root, file)
                rel_path  = os.path.relpath(full_path, extract_dir)
                arcname   = os.path.join("archive", rel_path) if not rel_path.startswith("archive") else rel_path
                z_out.write(full_path, arcname=arcname)

    return torch.load(fixed_pt, map_location='cpu', weights_only=False)

checkpoint = load_checkpoint_safe(PHASE1_MODEL_PATH)
print("Loaded checkpoint successfully.")

## 4. Custom ByteBPETokenizer

In [ ]:
SPECIALS = ['<pad>', '<unk>', '<s>', '</s>']
END_OF_WORD = '</w>'

class ByteBPETokenizer:
    def __init__(self, vocab_size=8000):
        self.vocab_size = vocab_size
        self.merges = []; self.merge_rank = {}; self.itos = []; self.stoi = {}

    @staticmethod
    def _word_to_symbols(word):
        return list(word.encode('utf-8').decode('latin-1')) + [END_OF_WORD]

    def _apply_merges_to_word(self, word):
        s = self._word_to_symbols(word)
        while len(s) > 1:
            p = [(s[i], s[i+1]) for i in range(len(s)-1)]
            c = [(self.merge_rank[x], i) for i, x in enumerate(p) if x in self.merge_rank]
            if not c: break
            _, i = min(c)
            s = s[:i] + [s[i]+s[i+1]] + s[i+2:]
        return s

    def encode(self, text, max_len=None):
        ids = [self.stoi.get(x, self.stoi['<unk>'])
               for w in text.split() for x in self._apply_merges_to_word(w)]
        return ids[:max_len] if max_len else ids

    def decode(self, ids):
        s = [self.itos[i] if i < len(self.itos) else '' for i in ids]
        words = [w for w in ''.join(s).split(END_OF_WORD) if w]
        return ' '.join(w.encode('latin-1').decode('utf-8', errors='replace') for w in words)

    @classmethod
    def load(cls, path):
        with open(path, 'r', encoding='utf-8') as f: d = json.load(f)
        t = cls(vocab_size=d['vocab_size'])
        t.itos = d['itos']; t.stoi = {x: i for i, x in enumerate(t.itos)}
        t.merges = [tuple(p) for p in d['merges']]
        t.merge_rank = {p: i for i, p in enumerate(t.merges)}
        return t

tokenizer = ByteBPETokenizer.load(TOKENIZER_PATH)
PAD_ID = tokenizer.stoi.get('<pad>', 0)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

## 5. Dataset Loader

In [ ]:
def load_docs(path):
    with open(path, 'r', encoding='utf-8') as f: data = json.load(f)
    if isinstance(data, list): return data
    elif isinstance(data, dict):
        if 'text' in data: return [data]
        docs = []
        for doc_id, body in data.items():
            if isinstance(body, list): docs.append({'id': doc_id, 'text': body})
            elif isinstance(body, dict):
                body['id'] = doc_id; docs.append(body)
        return docs
    return []

with open(LABEL_VOCAB_PATH, 'r', encoding='utf-8') as f: lv = json.load(f)
LABELS = lv if isinstance(lv, list) else list(lv.keys())
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL  = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABELS)

cl_train = load_docs(TRAIN_CL); cl_dev = load_docs(DEV_CL)
it_train = load_docs(TRAIN_IT); it_dev = load_docs(DEV_IT)
cl_test  = load_docs(TEST_CL);  it_test= load_docs(TEST_IT)

all_lbls = [l for doc in cl_train + it_train for l in doc.get('labels', [])]
counts = Counter(all_lbls); total = sum(counts.values())
class_weights = torch.tensor(
    [(total / (NUM_LABELS * max(counts.get(l, 1), 1))) ** 0.5 for l in LABELS],
    dtype=torch.float32
).to(device)
class_weights = class_weights / class_weights.mean()
print(f"Loaded {NUM_LABELS} labels. Class weights prepared.")

## 6. Architecture 4 with IA³ Hooks

In [ ]:
@dataclass
class ALiBiConfig:
    vocab_size: int
    d_model: int = 768
    n_layers: int = 12
    n_heads: int = 12
    d_ff: int = 2560
    dropout: float = 0.1

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps; self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

def get_alibi_slopes(n_heads):
    def gsp2(n):
        s = 2 ** (-(2 ** -(math.log2(n) - 3)))
        return [s * (s**i) for i in range(n)]
    if math.log2(n_heads) == int(math.log2(n_heads)):
        return torch.tensor(gsp2(n_heads))
    cp2 = 2 ** math.floor(math.log2(n_heads))
    return torch.tensor(gsp2(cp2) + gsp2(2*cp2)[0::2][:n_heads-cp2])

class ALiBiCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.n_heads = n_heads; self.dh = d_model // n_heads; self.dropout = dropout
        self.q_proj  = nn.Linear(d_model, d_model, bias=False)
        self.k_proj  = nn.Linear(d_model, d_model, bias=False)
        self.v_proj  = nn.Linear(d_model, d_model, bias=False)
        self.out_proj= nn.Linear(d_model, d_model, bias=False)
        self.q_norm  = RMSNorm(self.dh); self.k_norm = RMSNorm(self.dh)
        slopes = get_alibi_slopes(n_heads).unsqueeze(1).unsqueeze(2)
        self.register_buffer('slopes', slopes)
        self.ia3_k = None
        self.ia3_v = None

    def forward(self, x):
        B, S, _ = x.shape
        q = self.q_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.n_heads, self.dh).transpose(1, 2)
        q = self.q_norm(q); k = self.k_norm(k)
        if self.ia3_k is not None: k = k * self.ia3_k.view(1, self.n_heads, 1, self.dh)
        if self.ia3_v is not None: v = v * self.ia3_v.view(1, self.n_heads, 1, self.dh)
        pq = torch.arange(S, device=x.device).unsqueeze(1)
        pk = torch.arange(S, device=x.device).unsqueeze(0)
        alibi = self.slopes * (pk - pq).unsqueeze(0)
        causal = torch.triu(torch.full((S, S), float('-inf'), device=x.device), diagonal=1)
        attn_bias = alibi + causal.unsqueeze(0).unsqueeze(0)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_bias,
                dropout_p=self.dropout if self.training else 0.0)
        return self.out_proj(out.transpose(1,2).contiguous().view(B, S, self.n_heads * self.dh))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj   = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.drop = nn.Dropout(dropout)
        self.ia3_ff = None
    def forward(self, x):
        gate = F.silu(self.gate_proj(x)) * self.up_proj(x)
        if self.ia3_ff is not None: gate = gate * self.ia3_ff.unsqueeze(0).unsqueeze(0)
        return self.drop(self.down_proj(gate))

class ALiBiBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n1   = RMSNorm(cfg.d_model)
        self.attn = ALiBiCausalAttention(cfg.d_model, cfg.n_heads, cfg.dropout)
        self.n2   = RMSNorm(cfg.d_model)
        self.ffn  = SwiGLU(cfg.d_model, cfg.d_ff, cfg.dropout)
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.ffn(self.n2(x))
        return x

class ALiBiSwiGLULM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.drop    = nn.Dropout(cfg.dropout)
        self.blocks  = nn.ModuleList([ALiBiBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm_f  = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight
    def forward(self, idx):
        x = self.drop(self.tok_emb(idx))
        for b in self.blocks: x = b(x)
        return self.norm_f(x)

## 7. IA³ Module & CRF Layer

In [ ]:
class IA3Module(nn.Module):
    def __init__(self, backbone: ALiBiSwiGLULM):
        super().__init__()
        self.backbone = backbone
        d_model = backbone.cfg.d_model
        d_ff    = backbone.cfg.d_ff
        L       = backbone.cfg.n_layers

        self.l_k  = nn.ParameterList([nn.Parameter(torch.ones(d_model)) for _ in range(L)])
        self.l_v  = nn.ParameterList([nn.Parameter(torch.ones(d_model)) for _ in range(L)])
        self.l_ff = nn.ParameterList([nn.Parameter(torch.ones(d_ff)) for _ in range(L)])

    def attach_scales(self):
        for i, block in enumerate(self.backbone.blocks):
            block.attn.ia3_k = self.l_k[i]
            block.attn.ia3_v = self.l_v[i]
            block.ffn.ia3_ff = self.l_ff[i]

    def forward(self, idx):
        self.attach_scales()
        return self.backbone(idx)

class CRF(nn.Module):
    def __init__(self, num_tags: int):
        super().__init__()
        self.num_tags = num_tags
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags))
        self.start_transitions = nn.Parameter(torch.empty(num_tags))
        self.end_transitions   = nn.Parameter(torch.empty(num_tags))
        nn.init.uniform_(self.transitions, -0.1, 0.1)
        nn.init.uniform_(self.start_transitions, -0.1, 0.1)
        nn.init.uniform_(self.end_transitions, -0.1, 0.1)

    def neg_log_likelihood(self, emissions: torch.Tensor, tags: torch.Tensor) -> torch.Tensor:
        S = emissions.size(0)
        score = self.start_transitions[tags[0]] + emissions[0, tags[0]]
        for i in range(1, S): score += self.transitions[tags[i], tags[i-1]] + emissions[i, tags[i]]
        score += self.end_transitions[tags[-1]]
        alpha = self.start_transitions + emissions[0]
        for i in range(1, S):
            alpha = torch.logsumexp(alpha.unsqueeze(0) + self.transitions + emissions[i].unsqueeze(1), dim=1)
        log_Z = torch.logsumexp(alpha + self.end_transitions, dim=0)
        return log_Z - score

    def decode(self, emissions: torch.Tensor) -> List[int]:
        S, K = emissions.shape
        viterbi = torch.full((S, K), -1e9, device=emissions.device)
        backptr = torch.zeros((S, K), dtype=torch.long, device=emissions.device)
        viterbi[0] = self.start_transitions + emissions[0]
        for i in range(1, S):
            trans = viterbi[i-1].unsqueeze(1) + self.transitions
            viterbi[i], backptr[i] = trans.max(dim=0)
            viterbi[i] += emissions[i]
        viterbi[-1] += self.end_transitions
        best = [viterbi[-1].argmax().item()]
        for i in range(S-1, 0, -1): best.append(backptr[i, best[-1]].item())
        return list(reversed(best))

class IA3RRModel(nn.Module):
    def __init__(self, ia3: IA3Module, num_labels: int, dropout: float = 0.1):
        super().__init__()
        self.ia3 = ia3
        d_model = ia3.backbone.cfg.d_model
        self.bilstm = nn.LSTM(d_model, d_model // 2, num_layers=2, batch_first=True, bidirectional=True, dropout=dropout)
        self.drop = nn.Dropout(dropout)
        self.emission_head = nn.Linear(d_model, num_labels)
        self.crf = CRF(num_labels)

    def get_sent_embs(self, input_ids: torch.Tensor, attn_mask: torch.Tensor, chunk_size: int = 16):
        S = input_ids.size(0)
        sent_embs = []
        for i in range(0, S, chunk_size):
            c_ids = input_ids[i : i + chunk_size]
            c_mask = attn_mask[i : i + chunk_size]
            h = self.ia3(c_ids)
            m = c_mask.unsqueeze(-1).float()
            emb = (h * m).sum(1) / m.sum(1).clamp(min=1e-9)
            sent_embs.append(emb)
        return torch.cat(sent_embs, dim=0)

    def forward(self, input_ids, attn_mask, labels=None, chunk_size=16):
        embs = self.get_sent_embs(input_ids, attn_mask, chunk_size=chunk_size)
        x = embs.unsqueeze(0)
        lstm_out, _ = self.bilstm(x)
        emissions = self.emission_head(self.drop(lstm_out.squeeze(0)))
        if labels is not None:
            valid = labels >= 0
            if valid.sum() == 0: return torch.tensor(0.0, requires_grad=True, device=input_ids.device)
            return self.crf.neg_log_likelihood(emissions[valid], labels[valid])
        else:
            return self.crf.decode(emissions)

## 8. Initialize Model

In [ ]:
raw_cfg = checkpoint.get('config', {})
if hasattr(raw_cfg, '__dict__'): raw_cfg = raw_cfg.__dict__
d = raw_cfg if isinstance(raw_cfg, dict) else {}
cfg = ALiBiConfig(
    vocab_size=checkpoint.get('vocab_size', tokenizer.vocab_size),
    d_model=d.get('d_model', 768), n_layers=d.get('n_layers', 12),
    n_heads=d.get('n_heads', 12), d_ff=d.get('d_ff', 2560)
)

backbone = ALiBiSwiGLULM(cfg)
backbone.load_state_dict(checkpoint.get('model_state_dict', checkpoint), strict=False)
for p in backbone.parameters(): p.requires_grad = False

ia3 = IA3Module(backbone)
model = IA3RRModel(ia3, NUM_LABELS).to(device)

tot = sum(p.numel() for p in model.parameters())
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {tot:,} | Trainable: {tr:,} ({100*tr/tot:.3f}%)")

## 9. Dataset with Context Window

In [ ]:
class RRDataset(Dataset):
    def __init__(self, docs, tokenizer, label2id, max_seq_len, context_window=3, domain_prefix='', is_test=False):
        self.items = []
        pad_id = tokenizer.stoi.get('<pad>', 0)
        for doc in docs:
            sents = doc['text']
            labels = doc.get('labels', [None] * len(sents))
            ids_list, masks_list, lbls_list = [], [], []
            for i, (sent, lbl) in enumerate(zip(sents, labels)):
                s = max(0, i - context_window)
                e = min(len(sents), i + context_window + 1)
                ctx = f"{domain_prefix} " + " </s> ".join(sents[s:i])
                ctx += f" </s> {sent} </s> " + " </s> ".join(sents[i+1:e])
                ids = tokenizer.encode(ctx.strip(), max_len=max_seq_len)
                pad = max_seq_len - len(ids)
                ids_list.append(ids + [pad_id] * pad)
                masks_list.append([1] * len(ids) + [0] * pad)
                lbls_list.append(label2id.get(lbl, 0) if (not is_test and lbl) else -1)
            self.items.append({
                'doc_id': doc['id'],
                'input_ids': torch.tensor(ids_list, dtype=torch.long),
                'attention_mask': torch.tensor(masks_list, dtype=torch.long),
                'labels': torch.tensor(lbls_list, dtype=torch.long),
            })
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

collate_fn = lambda b: b

from torch.utils.data import ConcatDataset
train_ds = ConcatDataset([
    RRDataset(cl_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]"),
    RRDataset(it_train, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
])
dev_ds = ConcatDataset([
    RRDataset(cl_dev, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]"),
    RRDataset(it_dev, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]")
])
cl_te = RRDataset(cl_test, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[CL]", is_test=True)
it_te = RRDataset(it_test, tokenizer, LABEL2ID, MAX_SEQ_LEN, CONTEXT_WINDOW, "[IT]", is_test=True)

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=1, shuffle=False, collate_fn=collate_fn)
print(f"Train docs: {len(train_ds)} | Dev docs: {len(dev_ds)}")

## 10. Training Loop

In [ ]:
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=4, eta_min=1e-6)

def evaluate(model, loader):
    model.eval()
    all_p, all_l = [], []
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids, mask, lbls = doc['input_ids'].to(device), doc['attention_mask'].to(device), doc['labels'].to(device)
            with torch.amp.autocast('cuda'):
                preds = model(ids, mask, chunk_size=CHUNK_SIZE)
            all_p.extend(preds)
            all_l.extend(lbls.cpu().tolist())
    vm = [i for i, l in enumerate(all_l) if l >= 0]
    ap = [all_p[i] for i in vm]; al = [all_l[i] for i in vm]
    return (f1_score(al, ap, average='macro', zero_division=0),
            precision_score(al, ap, average='macro', zero_division=0),
            recall_score(al, ap, average='macro', zero_division=0))

best_f1 = 0.0
best_ckpt = os.path.join(OUTPUT_DIR, 'best_arch4_ia3.pt')
print("Starting IA3 training...")

for epoch in range(EPOCHS):
    model.train()
    ep_loss, steps = 0.0, 0
    for batch in train_loader:
        doc = batch[0]
        ids, mask, lbls = doc['input_ids'].to(device), doc['attention_mask'].to(device), doc['labels'].to(device)
        with torch.amp.autocast('cuda'):
            loss = model(ids, mask, labels=lbls, chunk_size=CHUNK_SIZE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        optimizer.step()
        ep_loss += loss.item()
        steps += 1

    scheduler.step()
    f1, p, r = evaluate(model, dev_loader)
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {ep_loss/max(1,steps):.4f} | Dev Macro-F1: {f1:.4f} (P: {p:.4f}, R: {r:.4f})")
    if f1 > best_f1:
        best_f1 = f1
        torch.save({'model_state_dict': model.state_dict(), 'f1': f1}, best_ckpt)
        print(f"  -> Saved Best (Macro-F1: {best_f1:.4f})")

print(f"Peak Dev Macro-F1: {best_f1:.4f}")

## 11. Predictions & Codabench Submission

In [ ]:
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

def gen_preds(model, test_ds):
    loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_fn)
    preds = {}
    with torch.no_grad():
        for batch in loader:
            doc = batch[0]
            ids, mask = doc['input_ids'].to(device), doc['attention_mask'].to(device)
            with torch.amp.autocast('cuda'):
                pred_ids = model(ids, mask, chunk_size=CHUNK_SIZE)
            preds[doc['doc_id']] = [ID2LABEL[p] for p in pred_ids]
    return preds

cl_preds = gen_preds(model, cl_te)
it_preds = gen_preds(model, it_te)
all_preds = {**cl_preds, **it_preds}

pred_path = os.path.join(OUTPUT_DIR, 'predictions.json')
with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(all_preds, f, indent=2, ensure_ascii=False)

zip_path = '/kaggle/working/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, arcname='predictions.json')
print(f"Submission ready at: {zip_path}")